In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeRDDAnalysis") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark Version:", spark.version)

Spark Version: 3.5.0


In [2]:
rdd = sc.textFile("/workspace/employee_data.csv")

header = rdd.first()

employees = rdd.filter(lambda x: x != header) \
               .map(lambda x: x.split(","))

employees.collect()

[['1', 'Amit', 'IT', '55000'],
 ['2', 'Rahul', 'HR', '40000'],
 ['3', 'Neha', 'IT', '65000'],
 ['4', 'Priya', 'Finance', '70000'],
 ['5', 'Karan', 'IT', '50000'],
 ['6', 'Simran', 'HR', '45000'],
 ['7', 'Rohit', 'Finance', '60000']]

In [8]:
#Sort by Salary Descending
sorted_employees = employees.sortBy(
    lambda x: int(x[3]),
    ascending=False
)

print("Employees Sorted By Salary:")

for emp in sorted_employees.collect():
    print(emp)

Employees Sorted By Salary:
['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']
['1', 'Amit', 'IT', '55000']
['5', 'Karan', 'IT', '50000']
['6', 'Simran', 'HR', '45000']
['2', 'Rahul', 'HR', '40000']


In [9]:
#Department wise total
dept_totals = employees.map(
    lambda x: (x[2], int(x[3]))
).reduceByKey(
    lambda a, b: a + b
)

print("Department Salary Totals:")

for dept, total in dept_totals.collect():
    print(f"{dept}: {total}")

Department Salary Totals:
IT: 170000
HR: 85000
Finance: 130000


In [10]:
#top 3 highest
top3 = sorted_employees.take(3)

for emp in top3:
    print(emp)

['4', 'Priya', 'Finance', '70000']
['3', 'Neha', 'IT', '65000']
['7', 'Rohit', 'Finance', '60000']


In [11]:
#Saving
import os

os.makedirs("/workspace/output", exist_ok=True)

with open("/workspace/output/top_3_employees.txt", "w") as f:
    f.write("id,name,department,salary\n")

    for emp in top3:
        f.write(",".join(emp) + "\n")

print("File saved successfully.")

File saved successfully.
